# Dual-Agent Ensemble Evaluation — P1 (Q2 INASS Paper)

Notebook ini mengevaluasi **5 strategi agregasi** untuk dual-agent ensemble:
- **M1 (NV-Reason-CXR-3B)** sebagai **Detector** — high sensitivity, over-predicts
- **M2 (ChestXReasoner)** sebagai **Verifier** — high specificity, under-predicts

**Strategi:**
| ID | Nama | Logika |
|---|---|---|
| S1 | Unanimous Positive | PTX hanya jika M1 DAN M2 setuju |
| S2 | Any Positive | PTX jika M1 ATAU M2 predict PTX |
| S3 | Detector→Verifier | M1 screen → jika positif, M2 verify |
| S4 | PBR-Guided Weighted | Weight berdasarkan jarak PBR ke ideal |
| S5 | Confidence-Weighted | Weight berdasarkan certainty jawaban model |

**Data:** Reuses Exp1 v2 results (M1: 11,950 records, M2: 11,950 records). Tidak perlu inference baru.

## 1 — Setup & Imports

In [ ]:
import sys
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path('src').resolve()))
from ensemble import (
    load_paired, run_ensemble, run_single_model_baselines,
    compare_all, compute_pbr, compute_sdr, compute_pbr_table,
    STRATEGIES, STRATEGY_FNS,
)
from evaluate import extract_diagnosis, compute_classification_metrics

# Style
matplotlib.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_palette('colorblind')

# Paths
M1_PATH = Path('result-M1/all_results.jsonl')
M2_PATH = Path('result-M2/all_results.jsonl')
FIGURES_DIR = Path('figures-ensemble')
RESULTS_DIR = Path('results-ensemble')
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

print('Setup complete.')

## 2 — Load & Pair Data

In [ ]:
df_paired = load_paired(M1_PATH, M2_PATH)

print(f'\nShape: {df_paired.shape}')
print(f'Columns: {list(df_paired.columns)}')
print(f'k_shots: {sorted(df_paired["k_shot"].unique())}')
print(f'seeds: {sorted(df_paired["seed"].unique())}')
print(f'test_sets: {df_paired["test_set"].unique().tolist()}')
print(f'\nGround truth distribution:')
print(df_paired.groupby('test_set')['ground_truth'].value_counts().unstack(fill_value=0))
print(f'\nPrediction distribution M1:')
print(df_paired.groupby('test_set')['pred_m1'].value_counts().unstack(fill_value=0))
print(f'\nPrediction distribution M2:')
print(df_paired.groupby('test_set')['pred_m2'].value_counts().unstack(fill_value=0))

## 3 — Quick Sanity Check: Agreement M1 vs M2

In [ ]:
# Agreement per k-shot
print('Agreement Rate M1 vs M2 per (k_shot, test_set):')
print('=' * 70)
for (k, ts), grp in df_paired.groupby(['k_shot', 'test_set']):
    agree = (grp['pred_m1'] == grp['pred_m2']).mean()
    both_ptx = ((grp['pred_m1'] == 1) & (grp['pred_m2'] == 1)).mean()
    both_norm = ((grp['pred_m1'] == 0) & (grp['pred_m2'] == 0)).mean()
    m1_only = ((grp['pred_m1'] == 1) & (grp['pred_m2'] == 0)).mean()
    m2_only = ((grp['pred_m1'] == 0) & (grp['pred_m2'] == 1)).mean()
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'  k={k} {short_ts:>8s} | '
          f'Agree={agree:.3f}  Both_PTX={both_ptx:.3f}  Both_Norm={both_norm:.3f}  '
          f'M1_only={m1_only:.3f}  M2_only={m2_only:.3f}')

## 4 — PBR Table (M1 & M2 Individual)

In [ ]:
pbr_table = compute_pbr_table(df_paired)
print('PBR per (k_shot, test_set) — pooled over seeds:')
print('  PBR=1.0 is ideal. >1 = over-predict, <1 = under-predict')
print('=' * 60)
for _, row in pbr_table.iterrows():
    short_ts = 'Internal' if 'Internal' in row['test_set'] else 'External'
    print(f"  k={int(row['k_shot'])} {short_ts:>8s} | "
          f"PBR_M1={row['pbr_m1']:.4f}  PBR_M2={row['pbr_m2']:.4f}")
print()
pbr_table

## 5 — Run Ensemble Evaluation (All Strategies + Baselines)

In [ ]:
# Pooled over seeds (for main results)
df_results = compare_all(df_paired, per_seed=False)
df_results.to_csv(RESULTS_DIR / 'ensemble_results_pooled.csv', index=False)

print(f'Total rows: {len(df_results)}')
print(f'Strategies: {df_results["strategy"].unique().tolist()}')
print()

# Show key columns
display_cols = ['strategy', 'k_shot', 'test_set', 'sensitivity', 'specificity',
                'precision', 'f1', 'accuracy', 'pbr', 'sdr']
df_results[display_cols]

In [ ]:
# Per-seed (for variance analysis & statistical tests)
df_per_seed = compare_all(df_paired, per_seed=True)
df_per_seed.to_csv(RESULTS_DIR / 'ensemble_results_per_seed.csv', index=False)

print(f'Per-seed rows: {len(df_per_seed)}')
df_per_seed.head(10)

## 6 — Summary Table: Mean ± Std per Strategy

In [ ]:
METRIC_COLS = ['sensitivity', 'specificity', 'precision', 'f1', 'accuracy', 'pbr', 'sdr']

# Compute mean ± std across seeds
summary_rows = []
for (strat, k, ts), grp in df_per_seed.groupby(['strategy', 'k_shot', 'test_set']):
    row = {'strategy': strat, 'k_shot': k, 'test_set': ts}
    for col in METRIC_COLS:
        vals = grp[col].dropna()
        row[f'{col}_mean'] = vals.mean()
        row[f'{col}_std'] = vals.std()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / 'ensemble_summary_mean_std.csv', index=False)

# Pretty print for Internal test set
for ts in ['CheXpert Plus (Internal)', 'RS UNAIR (External)']:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n{"=" * 100}')
    print(f'  {short_ts} — Mean ± Std across 5 seeds')
    print(f'{"=" * 100}')
    sub = summary_df[summary_df['test_set'] == ts].sort_values(['k_shot', 'strategy'])
    for k in sorted(sub['k_shot'].unique()):
        print(f'\n  k={int(k)}:')
        k_sub = sub[sub['k_shot'] == k]
        for _, r in k_sub.iterrows():
            print(f"    {r['strategy']:<20s} | "
                  f"F1={r['f1_mean']:.3f}±{r['f1_std']:.3f}  "
                  f"Sens={r['sensitivity_mean']:.3f}±{r['sensitivity_std']:.3f}  "
                  f"Spec={r['specificity_mean']:.3f}±{r['specificity_std']:.3f}  "
                  f"PBR={r['pbr_mean']:.3f}  "
                  f"SDR={r['sdr_mean']:.1f}%")

## 7 — Figure 1: F1 Score Comparison (Single vs Ensemble per k-shot)

In [ ]:
STRATEGY_LABELS = {
    'M1_single': 'M1 (Detector)',
    'M2_single': 'M2 (Verifier)',
    'S1_unanimous': 'S1: Unanimous',
    'S2_any': 'S2: Any Positive',
    'S3_detect_verify': 'S3: Detect→Verify',
    'S4_pbr_weighted': 'S4: PBR-Weighted',
    'S5_conf_weighted': 'S5: Conf-Weighted',
}

STRATEGY_COLORS = {
    'M1_single': '#E53935',
    'M2_single': '#1E88E5',
    'S1_unanimous': '#43A047',
    'S2_any': '#FB8C00',
    'S3_detect_verify': '#8E24AA',
    'S4_pbr_weighted': '#00ACC1',
    'S5_conf_weighted': '#6D4C41',
}

STRATEGY_MARKERS = {
    'M1_single': 'o',
    'M2_single': 's',
    'S1_unanimous': '^',
    'S2_any': 'v',
    'S3_detect_verify': 'D',
    'S4_pbr_weighted': 'P',
    'S5_conf_weighted': 'X',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
test_sets = ['CheXpert Plus (Internal)', 'RS UNAIR (External)']

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single'] + STRATEGIES:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        linestyle = '--' if '_single' in strat else '-'
        ax.errorbar(
            s['k_shot'], s['f1_mean'], yerr=s['f1_std'],
            label=STRATEGY_LABELS.get(strat, strat),
            marker=STRATEGY_MARKERS.get(strat, 'o'),
            color=STRATEGY_COLORS.get(strat),
            capsize=3, linewidth=2, markersize=7,
            linestyle=linestyle,
        )
    
    short_ts = 'Internal (CheXpert)' if 'Internal' in ts else 'External (RS UNAIR)'
    ax.set_xlabel('k per class')
    ax.set_ylabel('F1 Score')
    ax.set_title(short_ts)
    ax.set_xticks(sorted(df_paired['k_shot'].unique()))
    ax.legend(fontsize=7, loc='best')
    ax.set_ylim(0, 1.0)

plt.suptitle('F1 Score: Single Model vs Ensemble Strategies', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_f1_comparison.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_f1_comparison.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig1_f1_comparison')

## 8 — Figure 2: Sensitivity & Specificity per Strategy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single'] + STRATEGIES:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        label = STRATEGY_LABELS.get(strat, strat)
        color = STRATEGY_COLORS.get(strat)
        # Sensitivity as solid, Specificity as dashed
        ax.plot(s['k_shot'], s['sensitivity_mean'], '-',
                marker=STRATEGY_MARKERS.get(strat, 'o'),
                color=color, linewidth=1.5, markersize=5, alpha=0.9)
        ax.plot(s['k_shot'], s['specificity_mean'], '--',
                marker=STRATEGY_MARKERS.get(strat, 'o'),
                color=color, linewidth=1.5, markersize=5, alpha=0.5)
    
    # Custom legend: one entry per strategy + sens/spec indicator
    from matplotlib.lines import Line2D
    handles = []
    for strat in ['M1_single', 'M2_single'] + STRATEGIES:
        handles.append(Line2D([0], [0], color=STRATEGY_COLORS.get(strat),
                              marker=STRATEGY_MARKERS.get(strat, 'o'),
                              label=STRATEGY_LABELS.get(strat, strat),
                              linewidth=1.5, markersize=5))
    handles.append(Line2D([0], [0], color='gray', linestyle='-', label='Sensitivity (solid)'))
    handles.append(Line2D([0], [0], color='gray', linestyle='--', label='Specificity (dashed)'))
    
    short_ts = 'Internal (CheXpert)' if 'Internal' in ts else 'External (RS UNAIR)'
    ax.set_xlabel('k per class')
    ax.set_ylabel('Score')
    ax.set_title(short_ts)
    ax.set_xticks(sorted(df_paired['k_shot'].unique()))
    ax.legend(handles=handles, fontsize=6, loc='best')
    ax.set_ylim(0, 1.05)

plt.suptitle('Sensitivity (solid) & Specificity (dashed) per Strategy', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_sens_spec_comparison.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig2_sens_spec_comparison.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig2_sens_spec_comparison')

## 9 — Figure 3: PBR Trajectory (Closer to 1.0 = Better)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single'] + STRATEGIES:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        linestyle = '--' if '_single' in strat else '-'
        ax.plot(
            s['k_shot'], s['pbr_mean'],
            label=STRATEGY_LABELS.get(strat, strat),
            marker=STRATEGY_MARKERS.get(strat, 'o'),
            color=STRATEGY_COLORS.get(strat),
            linewidth=2, markersize=7,
            linestyle=linestyle,
        )
    
    # Ideal PBR line
    ax.axhline(y=1.0, color='black', linewidth=1.5, linestyle=':', alpha=0.7, label='Ideal (PBR=1.0)')
    
    short_ts = 'Internal (CheXpert)' if 'Internal' in ts else 'External (RS UNAIR)'
    ax.set_xlabel('k per class')
    ax.set_ylabel('PBR (Prediction Bias Ratio)')
    ax.set_title(short_ts)
    ax.set_xticks(sorted(df_paired['k_shot'].unique()))
    ax.legend(fontsize=7, loc='best')
    ax.set_ylim(0, 2.0)

plt.suptitle('PBR Trajectory — Closer to 1.0 is Better', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_pbr_trajectory.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig3_pbr_trajectory.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig3_pbr_trajectory')

## 10 — Figure 4: SDR Comparison (Lower = Better)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single'] + STRATEGIES:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        sdr_vals = s['sdr_mean'].fillna(0)
        linestyle = '--' if '_single' in strat else '-'
        ax.plot(
            s['k_shot'], sdr_vals,
            label=STRATEGY_LABELS.get(strat, strat),
            marker=STRATEGY_MARKERS.get(strat, 'o'),
            color=STRATEGY_COLORS.get(strat),
            linewidth=2, markersize=7,
            linestyle=linestyle,
        )
    
    ax.axhline(y=0, color='black', linewidth=1, linestyle=':', alpha=0.5)
    
    short_ts = 'Internal (CheXpert)' if 'Internal' in ts else 'External (RS UNAIR)'
    ax.set_xlabel('k per class')
    ax.set_ylabel('SDR (%) — Specificity Drop Rate')
    ax.set_title(short_ts)
    ax.set_xticks(sorted(df_paired['k_shot'].unique()))
    ax.legend(fontsize=7, loc='best')

plt.suptitle('SDR — Lower is Better (0% = no specificity loss)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_sdr_comparison.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig4_sdr_comparison.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig4_sdr_comparison')

## 11 — Figure 5: Bar Chart — Best k per Strategy (F1)

In [ ]:
# Find best k per strategy per test set
best_rows = []
for (strat, ts), grp in summary_df.groupby(['strategy', 'test_set']):
    best = grp.loc[grp['f1_mean'].idxmax()]
    best_rows.append({
        'strategy': strat,
        'test_set': ts,
        'best_k': int(best['k_shot']),
        'f1_mean': best['f1_mean'],
        'f1_std': best['f1_std'],
        'sensitivity_mean': best['sensitivity_mean'],
        'specificity_mean': best['specificity_mean'],
        'pbr_mean': best['pbr_mean'],
        'sdr_mean': best['sdr_mean'],
    })
best_df = pd.DataFrame(best_rows)
best_df.to_csv(RESULTS_DIR / 'best_k_per_strategy.csv', index=False)

print('Best k per strategy (by F1):')
print(best_df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

all_strats = ['M1_single', 'M2_single'] + STRATEGIES

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = best_df[best_df['test_set'] == ts].set_index('strategy').loc[all_strats].reset_index()
    
    x = np.arange(len(sub))
    bars = ax.bar(
        x, sub['f1_mean'], yerr=sub['f1_std'],
        color=[STRATEGY_COLORS.get(s, '#999') for s in sub['strategy']],
        capsize=3, edgecolor='white', linewidth=0.5,
    )
    
    # Annotate with best_k and F1 value
    for i, (_, r) in enumerate(sub.iterrows()):
        ax.text(i, r['f1_mean'] + r['f1_std'] + 0.02,
                f"k={int(r['best_k'])}\n{r['f1_mean']:.3f}",
                ha='center', fontsize=8)
    
    labels = [STRATEGY_LABELS.get(s, s) for s in sub['strategy']]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Best F1 Score')
    short_ts = 'Internal (CheXpert)' if 'Internal' in ts else 'External (RS UNAIR)'
    ax.set_title(short_ts)
    ax.set_ylim(0, 1.0)

plt.suptitle('Best F1 per Strategy (at optimal k)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_best_f1_bar.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig5_best_f1_bar.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig5_best_f1_bar')

## 12 — Figure 6: Scatter Plot — Sensitivity vs Specificity Trade-off

In [ ]:
# One point per (strategy, k) — shows the sens-spec trade-off space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, ts in enumerate(test_sets):
    ax = axes[idx]
    sub = summary_df[summary_df['test_set'] == ts]
    
    for strat in ['M1_single', 'M2_single'] + STRATEGIES:
        s = sub[sub['strategy'] == strat].sort_values('k_shot')
        if len(s) == 0:
            continue
        ax.scatter(
            s['specificity_mean'], s['sensitivity_mean'],
            label=STRATEGY_LABELS.get(strat, strat),
            marker=STRATEGY_MARKERS.get(strat, 'o'),
            color=STRATEGY_COLORS.get(strat),
            s=80, alpha=0.8, edgecolors='white', linewidth=0.5,
        )
        # Connect points with lines (k progression)
        ax.plot(s['specificity_mean'], s['sensitivity_mean'],
                color=STRATEGY_COLORS.get(strat), alpha=0.3, linewidth=1)
        
        # Annotate best-k point
        best_idx = s['f1_mean'].idxmax()
        best_row = s.loc[best_idx]
        ax.annotate(f"k={int(best_row['k_shot'])}",
                    (best_row['specificity_mean'], best_row['sensitivity_mean']),
                    fontsize=6, alpha=0.7)
    
    # Ideal point
    ax.scatter([1.0], [1.0], marker='*', color='gold', s=200, zorder=5,
              edgecolors='black', label='Ideal (1.0, 1.0)')
    
    ax.set_xlabel('Specificity')
    ax.set_ylabel('Sensitivity')
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    ax.set_title(short_ts)
    ax.legend(fontsize=6, loc='lower left')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Sensitivity vs Specificity Trade-off Space', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_sens_spec_scatter.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig6_sens_spec_scatter.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig6_sens_spec_scatter')

## 13 — Figure 7: Agreement Rate & Conflict Resolution

In [ ]:
# Use pooled results for agreement/conflict analysis
ens_only = df_results[~df_results['strategy'].str.contains('single')].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Agreement rate per k
ax = axes[0]
for ts in test_sets:
    sub = ens_only[ens_only['test_set'] == ts]
    # Agreement is same for all strategies at same k, so take S1
    s1 = sub[sub['strategy'] == 'S1_unanimous'].sort_values('k_shot')
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    ax.plot(s1['k_shot'], s1['agreement_rate'], '-o', label=short_ts, linewidth=2, markersize=8)

ax.set_xlabel('k per class')
ax.set_ylabel('Agreement Rate (M1 == M2)')
ax.set_title('(a) M1-M2 Agreement Rate')
ax.set_xticks(sorted(df_paired['k_shot'].unique()))
ax.legend()
ax.set_ylim(0, 1.05)

# (b) Conflict resolution accuracy per strategy (at best k)
ax = axes[1]
ts_internal = 'CheXpert Plus (Internal)'
sub = ens_only[ens_only['test_set'] == ts_internal]

strat_order = STRATEGIES
k_shots = sorted(df_paired['k_shot'].unique())
x = np.arange(len(k_shots))
bar_w = 0.15

for i, strat in enumerate(strat_order):
    s = sub[sub['strategy'] == strat].sort_values('k_shot')
    vals = s['conflict_resolution_acc'].fillna(0).values
    ax.bar(x + i * bar_w, vals, bar_w,
           label=STRATEGY_LABELS.get(strat, strat),
           color=STRATEGY_COLORS.get(strat),
           edgecolor='white', linewidth=0.5)

ax.set_xlabel('k per class')
ax.set_ylabel('Conflict Resolution Accuracy')
ax.set_title('(b) When M1≠M2, How Often is Ensemble Correct? (Internal)')
ax.set_xticks(x + bar_w * 2)
ax.set_xticklabels([str(k) for k in k_shots])
ax.legend(fontsize=7, loc='best')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_agreement_conflict.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig7_agreement_conflict.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig7_agreement_conflict')

## 14 — Figure 8: Heatmap — All Metrics at Best-k (Internal)

In [ ]:
# Heatmap of metrics at best-k for each strategy (Internal test set)
ts_internal = 'CheXpert Plus (Internal)'
heatmap_metrics = ['f1', 'sensitivity', 'specificity', 'precision', 'accuracy']

best_internal = best_df[best_df['test_set'] == ts_internal].copy()
all_strats = ['M1_single', 'M2_single'] + STRATEGIES
best_internal = best_internal.set_index('strategy').loc[all_strats].reset_index()

# Build matrix for heatmap
heat_data = best_internal[['sensitivity_mean', 'specificity_mean', 'f1_mean', 'pbr_mean']].values
heat_labels = ['Sensitivity', 'Specificity', 'F1', 'PBR']
row_labels = [STRATEGY_LABELS.get(s, s) + f" (k={int(r['best_k'])})" 
              for s, (_, r) in zip(best_internal['strategy'], best_internal.iterrows())]

# Annotations
annot = np.array([[f"{v:.3f}" for v in row] for row in heat_data])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    heat_data, annot=annot, fmt='',
    xticklabels=heat_labels,
    yticklabels=row_labels,
    cmap='YlOrRd', vmin=0.0, vmax=1.5,
    ax=ax, annot_kws={'fontsize': 11},
    linewidths=0.5,
)
ax.set_title('Metrics at Best-k per Strategy — Internal (CheXpert)', fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig8_heatmap_best_k.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig8_heatmap_best_k.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig8_heatmap_best_k')

## 15 — Figure 9: Cross-Institutional Robustness (Internal vs External)

In [ ]:
# F1 gap between internal and external for each strategy at best-k
gap_rows = []
for strat in all_strats:
    int_best = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == 'CheXpert Plus (Internal)')]
    ext_best = best_df[(best_df['strategy'] == strat) & (best_df['test_set'] == 'RS UNAIR (External)')]
    if len(int_best) > 0 and len(ext_best) > 0:
        gap_rows.append({
            'strategy': strat,
            'f1_internal': int_best.iloc[0]['f1_mean'],
            'f1_external': ext_best.iloc[0]['f1_mean'],
            'f1_gap': int_best.iloc[0]['f1_mean'] - ext_best.iloc[0]['f1_mean'],
        })

gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(RESULTS_DIR / 'cross_institutional_gap.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(gap_df))
w = 0.35

ax.bar(x - w/2, gap_df['f1_internal'], w,
       label='Internal (CheXpert)', color='#2196F3',
       edgecolor='white', linewidth=0.5)
ax.bar(x + w/2, gap_df['f1_external'], w,
       label='External (RS UNAIR)', color='#FF5722',
       edgecolor='white', linewidth=0.5)

# Annotate gap
for i, (_, r) in enumerate(gap_df.iterrows()):
    y_max = max(r['f1_internal'], r['f1_external'])
    ax.text(i, y_max + 0.02, f"gap={r['f1_gap']:+.3f}",
            ha='center', fontsize=7, style='italic')

labels = [STRATEGY_LABELS.get(s, s) for s in gap_df['strategy']]
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Best F1 Score')
ax.set_title('Cross-Institutional Robustness: Internal vs External')
ax.legend()
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig9_cross_institutional.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig9_cross_institutional.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig9_cross_institutional')

## 16 — Figure 10: Confusion Matrices — Best Strategy vs Single Models

In [ ]:
# Identify the best ensemble strategy (by F1 on internal)
int_best = best_df[best_df['test_set'] == 'CheXpert Plus (Internal)']
ens_best = int_best[~int_best['strategy'].str.contains('single')]
best_strat = ens_best.loc[ens_best['f1_mean'].idxmax(), 'strategy']
best_strat_k = int(ens_best.loc[ens_best['f1_mean'].idxmax(), 'best_k'])
print(f'Best ensemble strategy: {best_strat} at k={best_strat_k}')

# Get confusion matrices
from ensemble import STRATEGY_FNS

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

configs = [
    # (title, strategy, k, row, col)
    ('M1 Single', 'M1_single', best_strat_k, 0, 0),
    ('M2 Single', 'M2_single', best_strat_k, 0, 1),
    (f'{STRATEGY_LABELS[best_strat]}', best_strat, best_strat_k, 0, 2),
]

# Add all 5 ensemble strategies in bottom row? Let's do best 3 for clarity.
# Actually let's show M1, M2, and the 5 ensemble strategies at the best k.
# Rearrange: 2x4 or just show top 3 ensemble + baselines

ts_internal = 'CheXpert Plus (Internal)'
sub_k = df_paired[
    (df_paired['k_shot'] == best_strat_k) & 
    (df_paired['test_set'] == ts_internal)
]

plot_configs = [
    ('M1 (Detector)', 'pred_m1', 0, 0),
    ('M2 (Verifier)', 'pred_m2', 0, 1),
]

# Add ensemble predictions
from ensemble import s1_unanimous, s2_any, s3_detect_verify, s4_pbr_weighted, s5_conf_weighted

pred_m1 = sub_k['pred_m1'].values
pred_m2 = sub_k['pred_m2'].values
gt = sub_k['ground_truth'].values

# Get PBR for this k
pbr_m1 = compute_pbr(gt, pred_m1)
pbr_m2 = compute_pbr(gt, pred_m2)

ensemble_preds = {
    'S1: Unanimous': s1_unanimous(pred_m1, pred_m2),
    'S3: Detect→Verify': s3_detect_verify(pred_m1, pred_m2),
    'S4: PBR-Weighted': s4_pbr_weighted(pred_m1, pred_m2, pbr_m1=pbr_m1, pbr_m2=pbr_m2),
}

# Top row: M1, M2, best ensemble
# Bottom row: S1, S3, S4
all_configs = [
    ('M1 (Detector)', pred_m1, 0, 0),
    ('M2 (Verifier)', pred_m2, 0, 1),
    (f'{STRATEGY_LABELS[best_strat]} (Best)', 
     STRATEGY_FNS[best_strat](pred_m1, pred_m2, pbr_m1=pbr_m1, pbr_m2=pbr_m2,
                               conf_m1=sub_k['conf_m1'].values, conf_m2=sub_k['conf_m2'].values),
     0, 2),
]

# Add 3 more strategies for bottom row
bottom_strats = [s for s in STRATEGIES if s != best_strat][:3]
for col_idx, strat in enumerate(bottom_strats):
    pred = STRATEGY_FNS[strat](pred_m1, pred_m2, pbr_m1=pbr_m1, pbr_m2=pbr_m2,
                                conf_m1=sub_k['conf_m1'].values, conf_m2=sub_k['conf_m2'].values)
    all_configs.append((STRATEGY_LABELS[strat], pred, 1, col_idx))

for title, preds, row, col in all_configs:
    ax = axes[row, col]
    from sklearn.metrics import confusion_matrix as cm_fn, f1_score
    cm = cm_fn(gt, preds, labels=[0, 1])
    cm_pct = cm.astype(float) / cm.sum() * 100
    annot = np.array([
        [f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)' for j in range(2)]
        for i in range(2)
    ])
    sns.heatmap(
        cm, annot=annot, fmt='', cmap='Blues',
        xticklabels=['Normal', 'PTX'],
        yticklabels=['Normal', 'PTX'],
        ax=ax, annot_kws={'fontsize': 11},
    )
    f1 = f1_score(gt, preds, zero_division=0)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title(f'{title}\n(k={best_strat_k}, F1={f1:.3f})')

plt.suptitle(f'Confusion Matrices at k={best_strat_k} — Internal (CheXpert)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig10_confusion_matrices.png', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig10_confusion_matrices.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig10_confusion_matrices')

## 17 — Statistical Significance Tests

In [ ]:
# Test: Ensemble strategy vs best single model (Wilcoxon signed-rank per seed)
print('=' * 80)
print('STATISTICAL TESTS: Ensemble vs Single Model (Wilcoxon signed-rank, per seed)')
print('=' * 80)
print('H0: Tidak ada perbedaan F1 antara ensemble dan single model')
print('H1: Ada perbedaan F1\n')

for ts in test_sets:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n--- {short_ts} ---')
    ts_data = df_per_seed[df_per_seed['test_set'] == ts]
    
    for k in sorted(df_paired['k_shot'].unique()):
        k_data = ts_data[ts_data['k_shot'] == k]
        
        # Best single model F1 per seed
        m1_f1 = k_data[k_data['strategy'] == 'M1_single'].sort_values('seed')['f1'].values
        m2_f1 = k_data[k_data['strategy'] == 'M2_single'].sort_values('seed')['f1'].values
        
        if len(m1_f1) == 0 or len(m2_f1) == 0:
            continue
        
        # Use max(M1, M2) per seed as baseline
        best_single_f1 = np.maximum(m1_f1, m2_f1)
        
        print(f'  k={int(k)}:')
        for strat in STRATEGIES:
            ens_f1 = k_data[k_data['strategy'] == strat].sort_values('seed')['f1'].values
            if len(ens_f1) != len(best_single_f1):
                continue
            
            delta = ens_f1 - best_single_f1
            mean_delta = delta.mean()
            
            if len(set(delta)) > 1:
                stat, p = wilcoxon(ens_f1, best_single_f1)
            else:
                p = 1.0
            
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            label = STRATEGY_LABELS.get(strat, strat)
            print(f'    {label:<22s}  ΔF1={mean_delta:+.4f}  p={p:.4f} {sig}')

In [ ]:
# Test: Internal vs External per ensemble strategy (Mann-Whitney U)
print('\n' + '=' * 80)
print('STATISTICAL TESTS: Internal vs External per Strategy (Mann-Whitney U)')
print('=' * 80)

for strat in ['M1_single', 'M2_single'] + STRATEGIES:
    label = STRATEGY_LABELS.get(strat, strat)
    print(f'\n  {label}:')
    
    for k in sorted(df_paired['k_shot'].unique()):
        int_f1 = df_per_seed[
            (df_per_seed['strategy'] == strat) & 
            (df_per_seed['k_shot'] == k) &
            (df_per_seed['test_set'] == 'CheXpert Plus (Internal)')
        ]['f1'].values
        
        ext_f1 = df_per_seed[
            (df_per_seed['strategy'] == strat) & 
            (df_per_seed['k_shot'] == k) &
            (df_per_seed['test_set'] == 'RS UNAIR (External)')
        ]['f1'].values
        
        if len(int_f1) >= 3 and len(ext_f1) >= 3:
            stat, p = mannwhitneyu(int_f1, ext_f1, alternative='two-sided')
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            gap = np.mean(int_f1) - np.mean(ext_f1)
            print(f'    k={int(k)}: Int={np.mean(int_f1):.4f} Ext={np.mean(ext_f1):.4f} '
                  f'gap={gap:+.4f} p={p:.4f} {sig}')

## 18 — Net Improvement Analysis

In [ ]:
# Net improvement = F1_ensemble - max(F1_M1, F1_M2) per k
print('NET IMPROVEMENT: F1_ensemble - max(F1_M1, F1_M2)')
print('Positive = ensemble is better than best single model')
print('=' * 80)

for ts in test_sets:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    print(f'\n--- {short_ts} ---')
    sub = summary_df[summary_df['test_set'] == ts]
    
    for k in sorted(sub['k_shot'].unique()):
        k_sub = sub[sub['k_shot'] == k]
        m1_f1 = k_sub[k_sub['strategy'] == 'M1_single']['f1_mean'].values[0]
        m2_f1 = k_sub[k_sub['strategy'] == 'M2_single']['f1_mean'].values[0]
        best_single = max(m1_f1, m2_f1)
        
        print(f'  k={int(k)} (best_single={best_single:.4f}, M1={m1_f1:.4f}, M2={m2_f1:.4f}):')
        for strat in STRATEGIES:
            ens_f1 = k_sub[k_sub['strategy'] == strat]['f1_mean'].values
            if len(ens_f1) > 0:
                net = ens_f1[0] - best_single
                label = STRATEGY_LABELS.get(strat, strat)
                marker = '+' if net > 0 else '-' if net < 0 else '='
                print(f'    [{marker}] {label:<22s}  F1={ens_f1[0]:.4f}  net={net:+.4f}')

## 19 — Hypothesis Testing Summary

In [ ]:
print('=' * 80)
print('HYPOTHESIS TESTING SUMMARY')
print('=' * 80)

ts_int = 'CheXpert Plus (Internal)'
ts_ext = 'RS UNAIR (External)'

# H1: Ensemble F1 > single model
print('\n--- H1: Dual-agent ensemble > best single model (F1) ---')
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    sub = summary_df[summary_df['test_set'] == ts]
    for k in sorted(sub['k_shot'].unique()):
        k_sub = sub[sub['k_shot'] == k]
        m1_f1 = k_sub[k_sub['strategy'] == 'M1_single']['f1_mean'].values[0]
        m2_f1 = k_sub[k_sub['strategy'] == 'M2_single']['f1_mean'].values[0]
        best_single = max(m1_f1, m2_f1)
        best_ens_row = k_sub[~k_sub['strategy'].str.contains('single')].sort_values('f1_mean', ascending=False).iloc[0]
        better = best_ens_row['f1_mean'] > best_single
        print(f'  {short_ts} k={int(k)}: best_single={best_single:.4f}, '
              f'best_ensemble={best_ens_row["f1_mean"]:.4f} ({best_ens_row["strategy"]})'
              f' → {"SUPPORTED" if better else "NOT SUPPORTED"}')

# H2: PBR closer to 1.0
print('\n--- H2: Ensemble PBR closer to 1.0 than single models ---')
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    sub = summary_df[summary_df['test_set'] == ts]
    for k in sorted(sub['k_shot'].unique()):
        k_sub = sub[sub['k_shot'] == k]
        m1_pbr = k_sub[k_sub['strategy'] == 'M1_single']['pbr_mean'].values[0]
        m2_pbr = k_sub[k_sub['strategy'] == 'M2_single']['pbr_mean'].values[0]
        best_single_dist = min(abs(m1_pbr - 1), abs(m2_pbr - 1))
        
        ens_sub = k_sub[~k_sub['strategy'].str.contains('single')]
        best_ens = ens_sub.loc[(ens_sub['pbr_mean'] - 1).abs().idxmin()]
        ens_dist = abs(best_ens['pbr_mean'] - 1)
        
        better = ens_dist < best_single_dist
        print(f'  {short_ts} k={int(k)}: single_best_dist={best_single_dist:.4f}, '
              f'ens_dist={ens_dist:.4f} ({best_ens["strategy"]})'
              f' → {"SUPPORTED" if better else "NOT SUPPORTED"}')

# H3: Ensemble SDR < M1 SDR
print('\n--- H3: Ensemble SDR < M1 SDR (ensemble protects specificity) ---')
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    sub = summary_df[summary_df['test_set'] == ts]
    k_max = max(sub['k_shot'].unique())
    k_sub = sub[sub['k_shot'] == k_max]
    
    m1_sdr = k_sub[k_sub['strategy'] == 'M1_single']['sdr_mean'].values[0]
    for strat in STRATEGIES:
        ens_sdr = k_sub[k_sub['strategy'] == strat]['sdr_mean'].values
        if len(ens_sdr) > 0:
            better = ens_sdr[0] < m1_sdr
            label = STRATEGY_LABELS.get(strat, strat)
            print(f'  {short_ts} k={int(k_max)}: M1_SDR={m1_sdr:.1f}%, '
                  f'{label}_SDR={ens_sdr[0]:.1f}%'
                  f' → {"SUPPORTED" if better else "NOT SUPPORTED"}')

## 20 — Export All Results

In [ ]:
# Summary of exported files
print('EXPORTED CSV FILES:')
for f in sorted(RESULTS_DIR.glob('*.csv')):
    print(f'  {f.name}')

print(f'\nEXPORTED FIGURES:')
for f in sorted(FIGURES_DIR.glob('fig*')):
    print(f'  {f.name}')

print(f'\nTotal figures: {len(list(FIGURES_DIR.glob("fig*.png")))}')
print(f'Total CSV files: {len(list(RESULTS_DIR.glob("*.csv")))}')

## 21 — Key Findings Summary

In [ ]:
print('=' * 80)
print('KEY FINDINGS — Dual-Agent Ensemble (P1)')
print('=' * 80)

# Overall best
for ts in [ts_int, ts_ext]:
    short_ts = 'Internal' if 'Internal' in ts else 'External'
    sub = summary_df[summary_df['test_set'] == ts]
    
    # Best single
    singles = sub[sub['strategy'].str.contains('single')]
    best_single = singles.loc[singles['f1_mean'].idxmax()]
    
    # Best ensemble  
    ensembles = sub[~sub['strategy'].str.contains('single')]
    best_ens = ensembles.loc[ensembles['f1_mean'].idxmax()]
    
    improvement = best_ens['f1_mean'] - best_single['f1_mean']
    
    print(f'\n--- {short_ts} ---')
    print(f'  Best single model: {best_single["strategy"]} at k={int(best_single["k_shot"])} '
          f'(F1={best_single["f1_mean"]:.4f}±{best_single["f1_std"]:.4f})')
    print(f'  Best ensemble:     {best_ens["strategy"]} at k={int(best_ens["k_shot"])} '
          f'(F1={best_ens["f1_mean"]:.4f}±{best_ens["f1_std"]:.4f})')
    print(f'  Net improvement:   {improvement:+.4f} F1')
    print(f'  PBR (ensemble):    {best_ens["pbr_mean"]:.4f} (ideal=1.0)')
    print(f'  SDR (ensemble):    {best_ens["sdr_mean"]:.1f}% vs M1 SDR='
          f'{sub[(sub["strategy"]=="M1_single") & (sub["k_shot"]==best_ens["k_shot"])]["sdr_mean"].values[0]:.1f}%')

print('\n' + '=' * 80)
print('CONCLUSIONS:')
print('=' * 80)
print('  1. ???  (fill after running)')
print('  2. ???')
print('  3. ???')